In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

In [2]:
def load_data(path="../data/transformer_data.csv"):
    df = pd.read_csv(path)
    return df


In [3]:
def train_model(df):
    feature_cols = [
        "age_years", "load_factor", "maintenance_score",
        "oil_quality_index", "temperature_rise_c"
    ]
    X = df[feature_cols].values
    y = df["failure_within_1yr"].values

    # Held-out split so the reported metrics reflect generalization, not
    # memorization of the training rows.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_scaler = StandardScaler()
    X_train_scaled = train_scaler.fit_transform(X_train)
    X_test_scaled = train_scaler.transform(X_test)

    eval_model = RandomForestClassifier(random_state=42)
    eval_model.fit(X_train_scaled, y_train)  # <-- train the eval model

    y_test_scores = eval_model.predict_proba(X_test_scaled)[:, 1]
    y_test_pred = eval_model.predict(X_test_scaled)

    print("Held-out ROC-AUC:", roc_auc_score(y_test, y_test_scores))
    print("Held-out Average Precision:", average_precision_score(y_test, y_test_scores))
    print(classification_report(y_test, y_test_pred))

    # Refit on the full dataset for deployment scoring, now that the
    # held-out numbers above give an honest read on generalization.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(random_state=42)
    model.fit(X_scaled, y)

    y_scores = model.predict_proba(X_scaled)[:, 1]
    y_pred = model.predict(X_scaled)

    return model, scaler, feature_cols, y_scores, y_pred

In [4]:
def score_all_transformers(df, y_scores, y_pred):
    result = df.copy()
    result["failure_score"] = y_scores
    result["predicted_failure"] = y_pred
    result["tier"] = pd.qcut(result["failure_score"], q=3, labels=["low", "medium", "high"])
    result.to_csv("../data/transformer_failure_scores.csv", index=False)  # <-- fixed path
    return result.sort_values("failure_score", ascending=False)


In [5]:
df = load_data()
model, scaler, feature_cols, y_scores, y_pred = train_model(df)
scored = score_all_transformers(df, y_scores, y_pred)

scored.head()

Held-out ROC-AUC: 0.8668124268149883
Held-out Average Precision: 0.6234179820069271
              precision    recall  f1-score   support

           0       0.89      0.94      0.91       488
           1       0.66      0.48      0.56       112

    accuracy                           0.86       600
   macro avg       0.77      0.71      0.74       600
weighted avg       0.85      0.86      0.85       600



,transformer_id,feeder_id,age_years,load_factor,maintenance_score,oil_quality_index,temperature_rise_c,failure_within_1yr,failure_score,predicted_failure,tier
232,T0233,F030,30.2,0.599,0.352,0.460,73.8,1,1.0,1,high
2471,T2472,F149,26.5,0.570,0.281,0.373,69.8,1,1.0,1,high
780,T0781,F1317,10.3,0.640,0.144,0.294,60.0,1,1.0,1,high
1933,T1934,F1430,19.1,0.894,0.334,0.432,69.6,1,1.0,1,high
2374,T2375,F1049,28.3,0.729,0.370,0.388,72.8,1,1.0,1,high
